<a href="https://colab.research.google.com/github/SahanWijendra/Statistical-Learning-e22434/blob/main/assignments/Bayesian_Inference_Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Q. Bayesian Estimation of a User Ability Parameter from Item Responses

An online learning platform presents a user with a sequence of $n$ multiple-choice questions **one at a time**. Each question is either answered correctly or incorrectly, allowing the platform to update its estimate of the user's ability dynamically after every response.

Let $Y_i$ denote the user's response to the $i$-th item encountered:

$$Y_i=
\begin{cases}
1, & \text{if the user answers item } i \text{ correctly},\\
0, & \text{if the user answers item } i \text{ incorrectly}.
\end{cases}$$

The platform assumes that the probability of a correct response is governed by a two-parameter logistic (2PL) item response model. Specifically, conditional on the user's latent ability parameter $\Theta=\theta$, the response probability for item $i$ is:

$$P(Y_i=1\mid \Theta=\theta)=p_i(\theta)=\frac{1}{1+e^{-a_i(\theta-b_i)}},$$

where $a_i>0$ is the known discrimination parameter, and $b_i$ is the known difficulty parameter of item $i$.

Let $\mathbf{y}^{(k)} = (y_1, y_2, \dots, y_k)$ represent the **running vector of observed responses** up to the current step $k$ (where $1 \le k \le n$).

Before observing any responses, the platform initializes the user's latent ability estimate with a standard normal prior distribution:

$$f_{\Theta}^{(0)}(\theta) = \frac{1}{\sqrt{2\pi}} \exp\left(-\frac{\theta^2}{2}\right) \quad \text{implying} \quad \Theta \sim \mathscr{N}(0,1).$$

As the user progresses, the posterior distribution at step $k-1$ serves as the prior distribution for step $k$.

---

### Tasks

1. **Visualizing the Mechanics:** Plot $P(Y_i=1\mid \Theta=\theta)$ vs $\theta$ using Plotly for two distinct values of $a_i$, where one of those $a_i$ values is paired with three different difficulty values of $b_i$. Interpret how moving $b_i$ shifts the curve horizontally.
2. **Sequential Likelihood Contribution:** Write down the likelihood contribution $L(y_k \mid \theta)$ of a *single* new response $y_k$ at step $k$, given the latent ability $\theta$. Then, write down the joint likelihood function for the running history vector $\mathbf{y}^{(k)}$.
3. **Mathematical Formulation of the Running Update:** Write down the recursive relationship for the posterior density at step $k$, denoted $f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)})$, up to a proportionality constant, using the prior state $f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})$ and the new observation $y_k$.
4. **Dynamic Shifting:** Explain how a correct answer ($y_k = 1$) to a highly difficult item (large $b_k$) mathematically shifts the peak of the running posterior density distribution relative to the previous step.
5. **Tracking Certainty and Sharpness:** Explain how the discrimination parameter $a_k$ of the current item alters the variance (or "sharpness") of the distribution during a running update. What happens when $a_k$ is very large versus very small?
6. **Numerical Implementation of a Running Grid:** Describe a algorithmic approach to numerically approximate and maintain this running posterior density function on a fixed grid of $\theta$-values. Explicitly state how you would perform the sequential normalization step computationally after an item is answered.


7. **Evaluating Convergence over the Timeline:** Suppose the user's true, hidden latent ability is $\theta_{\text{true}} = 0.75$. Write a Python script that extends your previous grid simulation to track the performance of the running estimators over a sequence of $n = 20$ items.
* **Simulate Responses:** Dynamically generate the user's responses $y_k \in \{0, 1\}$ at each step by comparing a random draw from a Uniform distribution $U(0,1)$ against the true response probability $p_k(\theta_{\text{true}})$. Give each item a random difficulty $b_k \sim \mathscr{N}(0, 1)$ and a random discrimination $a_k \sim \text{Uniform}(0.5, 2.0)$.
* **Track Estimators:** At each step $k$, calculate and store the running Posterior Mean ($\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$) and the running Maximum A Posteriori ($\widehat{\theta}_{\mathrm{MAP}}^{(k)}$) estimate.
* **Visualize:** Use Plotly to create a single line chart showing the progression of both estimators from step $0$ to $20$. Add a static horizontal reference line at $y = 0.75$ representing $\theta_{\text{true}}$.
* **Analysis:** Briefly explain how the distance between your estimators and $\theta_{\text{true}}$ changes as $k$ increases, and interpret what this implies about the platform's confidence in its measurement.


## Sample Answer

### The 2PL Item Response Probability Model

Under the assumption of local independence (conditional independence of item responses given the latent ability parameter $\Theta = \theta$), the joint likelihood function for a sequence of observations $Y = y$ is expressed as:

$$L(\theta \mid y) = \prod_{i=1}^n p_i(\theta)^{y_i} \left(1 - p_i(\theta)\right)^{1 - y_i}$$

where the response probability for the $i$-th item follows a two-parameter logistic (2PL) model:

$$p_i(\theta) = \frac{1}{1 + \exp\left(-a_i(\theta - b_i)\right)}$$

In [ ]:
import numpy as np
import plotly.graph_objects as go

# 2-Parameter Logistic (2PL) Probability Calculation
def item_response_fn(ability, discrimination, difficulty):
    return 1.0 / (1.0 + np.exp(-discrimination * (ability - difficulty)))

# Latent ability spectrum (theta range)
ability_grid = np.linspace(-6, 6, 300)

# Curve configurations: distinct discrimination (a) and difficulty (b) parameters
item_specs = [
    {"a": 0.5, "b": 0, "dash_type": "dash"},
    {"a": 1.5, "b": -2, "dash_type": "solid"},
    {"a": 1.5, "b": 0, "dash_type": "solid"},
    {"a": 1.5, "b": 2, "dash_type": "solid"},
]

# Initialize interactive Plotly figure
fig = go.Figure()

for spec in item_specs:
    a_val = spec["a"]
    b_val = spec["b"]
    line_dash = spec["dash_type"]

    # Compute success probabilities across ability domain
    prob_values = item_response_fn(ability_grid, a_val, b_val)

    # Plot curve trace
    fig.add_trace(go.Scatter(
        x=ability_grid,
        y=prob_values,
        mode='lines',
        name=f"a = {a_val}, b = {b_val}",
        line=dict(dash=line_dash, width=2.5)
    ))

# Apply layout properties
fig.update_layout(
    title={
        'text': "Two-Parameter Logistic (2PL) Item Response Curves",
        'y': 0.9,
        'x': 0.5,
        'xanchor': 'center',
        'yanchor': 'top'
    },
    xaxis_title="Latent Ability (θ)",
    yaxis_title="Probability of Correct Response P(Y_i = 1 | θ)",
    xaxis=dict(range=[-6, 6], gridcolor='rgba(0,0,0,0.1)'),
    yaxis=dict(range=[0, 1.05], gridcolor='rgba(0,0,0,0.1)'),
    template="plotly_white",
    legend=dict(
        yanchor="top",
        y=0.95,
        xanchor="left",
        x=0.05,
        bgcolor="rgba(255,255,255,0.8)"
    )
)

# Render figure
fig.show()

### Posterior Distribution

For each step $k \in \{1, \dots, n\}$, given an observed item response $y_k$, the corresponding likelihood function is defined as:

$$L(y_k \mid \theta) = \left[p_k(\theta)\right]^{y_k} \left[1 - p_k(\theta)\right]^{1 - y_k}$$

The recursive update for the sequential posterior probability density function of $\Theta$, given the cumulative response history $\mathbf{y}^{(k)}$, is expressed by:

$$f_{\Theta\mid\mathbf{Y}^{(k)}}(\theta\mid\mathbf{y}^{(k)}) = \frac{ \left[ p_k(\theta)^{y_k} (1 - p_k(\theta))^{1 - y_k} \right] f_{\Theta\mid\mathbf{Y}^{(k-1)}}(\theta\mid\mathbf{y}^{(k-1)}) }{ \int_{-\infty}^{\infty} \left[ p_k(s)^{y_k} (1 - p_k(s))^{1 - y_k} \right] f_{\Theta\mid\mathbf{Y}^{(k-1)}}(s\mid\mathbf{y}^{(k-1)})\,ds }$$

**Term Definitions:**

* $f_{\Theta\mid\mathbf{Y}^{(k-1)}}(\theta\mid\mathbf{y}^{(k-1)})$ acts as the **prior distribution** at step $k$, taking the value of the posterior density obtained at step $k-1$.
* For the initial item ($k=1$), the process starts with a standard normal prior distribution:

$$f_{\Theta\mid\mathbf{Y}^{(0)}}(\theta\mid\mathbf{y}^{(0)}) = f_\Theta^{(0)}(\theta) = \frac{1}{\sqrt{2\pi}}\exp\left(-\frac{\theta^2}{2}\right)$$

* The denominator functions as a **normalizing factor**, integrating the product of the likelihood and current prior over the domain of $\theta$ to ensure that the total probability of the updated posterior density equals $1$.

### Baye's Estimate and the MAP Estimate

In a sequential updating framework, point estimators such as the **Posterior Mean** (the Bayes estimator minimizing squared-error loss) and the **Maximum A Posteriori (MAP)** estimate are recalculated at each iteration step $k$ based on the updated posterior density $f_{\Theta\mid\mathbf{Y}^{(k)}}(\theta\mid\mathbf{y}^{(k)})$.

1. Running Posterior Mean (Bayesian Expectation)
The Bayes estimate for latent ability at step $k$, derived by minimizing quadratic loss, represents the expectation of the active posterior distribution:

$$\widehat{\theta}_{\mathrm{Bayes}}^{(k)} = \mathbb{E}\left[\Theta \mid \mathbf{Y}^{(k)} = \mathbf{y}^{(k)}\right] = \int_{-\infty}^{\infty} \theta \, f_{\Theta\mid\mathbf{Y}^{(k)}}(\theta\mid\mathbf{y}^{(k)}) \, d\theta$$

2. Running Maximum A Posterior (MAP) Estimator
Concurrently, the most probable parameter value at step $k$ is given by the mode (highest peak) of the posterior distribution:

$$\widehat{\theta}_{\mathrm{MAP}}^{(k)} = \underset{\theta \in \mathbb{R}}{\operatorname{argmax}} \, f_{\Theta\mid\mathbf{Y}^{(k)}}(\theta\mid\mathbf{y}^{(k)})$$

**Implementation Note**

Computationally, as item responses accumulate, the evaluated grid array stores values for $f_{\Theta\mid\mathbf{Y}^{(k)}}(\theta\mid\mathbf{y}^{(k)})$:

* $\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$ is computed via trapezoidal integration over the grid: `np.trapezoid(theta * current_posterior, theta)`.
* $\widehat{\theta}_{\mathrm{MAP}}^{(k)}$ is identified by taking the coordinate at the peak probability index: `theta[np.argmax(current_posterior)]`.

### Numerical Implementation

In [ ]:
import numpy as np
import scipy.stats as stats
import plotly.graph_objects as go

# =====================================================================
# SECTION 1: DISCRETE BAYESIAN SEQUENTIAL UPDATE (4-ITEM DEMO)
# =====================================================================

# 1. Establish grid for latent ability domain
ability_axis = np.linspace(-5, 5, 500)

# 2. Base prior setup: Standard Normal N(0, 1)
prior_density = stats.norm.pdf(ability_axis, 0, 1)

# 3. 2PL Response Probability Calculation Function
def calculate_2pl_prob(theta_val, discrimination, difficulty):
    return 1.0 / (1.0 + np.exp(-discrimination * (theta_val - difficulty)))

# 4. Item encounter sequence data
item_sequence = [
    {"disc": 1.0, "diff": -1.5, "ans": 1},  # Step 1: Correct response to easy question
    {"disc": 1.5, "diff": 0.5,  "ans": 1},  # Step 2: Correct response to medium-hard question
    {"disc": 1.2, "diff": 1.5,  "ans": 0},  # Step 3: Incorrect response to hard question
    {"disc": 2.0, "diff": 0.2,  "ans": 1}   # Step 4: Correct response to highly discriminative question
]

# Instantiate Figure 1
fig1 = go.Figure()

# Plot baseline prior curve
fig1.add_trace(go.Scatter(
    x=ability_axis,
    y=prior_density,
    mode='lines',
    name='Initial Prior: N(0,1)',
    line=dict(dash='dash', width=2.5, color='gray')
))

# Iterative Bayesian belief updates
active_posterior = prior_density.copy()

for step_idx, item_info in enumerate(item_sequence):
    a_param = item_info["disc"]
    b_param = item_info["diff"]
    response = item_info["ans"]

    # Compute IRF across full ability domain
    item_prob = calculate_2pl_prob(ability_axis, a_param, b_param)

    # Evaluate single-item likelihood vector
    step_likelihood = (item_prob ** response) * ((1.0 - item_prob) ** (1 - response))

    # Calculate unnormalized posterior
    active_posterior *= step_likelihood

    # Trapezoidal numerical integration for normalization
    normalizing_factor = np.trapezoid(active_posterior, ability_axis)
    active_posterior /= normalizing_factor

    # Label formatting for plot trace
    outcome_str = "Correct" if response == 1 else "Incorrect"
    label = f"Step {step_idx+1}: Post-Item {step_idx+1} ({outcome_str}, a={a_param}, b={b_param})"

    # Append trace for updated posterior density
    fig1.add_trace(go.Scatter(
        x=ability_axis,
        y=active_posterior,
        mode='lines',
        name=label,
        line=dict(width=2)
    ))

# Figure 1 styling parameters
fig1.update_layout(
    title={
        'text': "Sequential Bayesian Update of User Ability (θ)",
        'y': 0.95, 'x': 0.5, 'xanchor': 'center', 'yanchor': 'top'
    },
    xaxis_title="Latent Ability Parameter (θ)",
    yaxis_title="Probability Density f(θ | y)",
    template="plotly_white",
    hovermode="x unified",
    legend=dict(
        yanchor="top", y=0.95, xanchor="left", x=0.02,
        bgcolor="rgba(255,255,255,0.7)"
    )
)

fig1.show()


# =====================================================================
# SECTION 2: CONVERGENCE TIMELINE SIMULATION (20 ITEMS)
# =====================================================================

# Seed setting for reproducible stochastic generation
np.random.seed(42)

# 1. Parameter Initialization
true_ability = 0.75
total_items = 20
grid_points = np.linspace(-5, 5, 1000)

# 2. Random parameter assignment for items
discrimination_array = np.random.uniform(0.5, 2.0, size=total_items)
difficulty_array = np.random.normal(0, 1, size=total_items)

# 3. Tracking initialization
bayes_estimates = [0.0]
map_estimates = [0.0]
timeline_steps = list(range(total_items + 1))

# Initialize prior state
sim_posterior = stats.norm.pdf(grid_points, 0, 1)

# 4. Execute 20-step simulation
for i in range(total_items):
    a_val = discrimination_array[i]
    b_val = difficulty_array[i]

    # Calculate probability at ground truth
    p_true = calculate_2pl_prob(true_ability, a_val, b_val)

    # Stochastic response draw
    user_response = 1 if np.random.uniform(0, 1) < p_true else 0

    # Grid likelihood calculation
    p_grid = calculate_2pl_prob(grid_points, a_val, b_val)
    grid_likelihood = (p_grid ** user_response) * ((1.0 - p_grid) ** (1 - user_response))

    # Sequential update & rescale
    sim_posterior *= grid_likelihood
    norm_const = np.trapezoid(sim_posterior, grid_points)
    sim_posterior /= norm_const

    # Compute point estimators
    est_bayes = np.trapezoid(grid_points * sim_posterior, grid_points)
    est_map = grid_points[np.argmax(sim_posterior)]

    bayes_estimates.append(est_bayes)
    map_estimates.append(est_map)

# 5. Build Figure 2
fig2 = go.Figure()

# Ground truth reference line
fig2.add_hline(
    y=true_ability,
    line_dash="dash",
    line_color="red",
    line_width=2,
    annotation_text=f"True Ability (θ = {true_ability})",
    annotation_position="bottom right"
)

# Trajectory traces
fig2.add_trace(go.Scatter(
    x=timeline_steps, y=bayes_estimates,
    mode='lines+markers',
    name='Posterior Mean (θ̂_Bayes)',
    line=dict(color='blue', width=2.5),
    marker=dict(size=6)
))

fig2.add_trace(go.Scatter(
    x=timeline_steps, y=map_estimates,
    mode='lines+markers',
    name='MAP Estimate (θ̂_MAP)',
    line=dict(color='green', width=2),
    marker=dict(size=6, symbol='square')
))

# Layout config
fig2.update_layout(
    title={
        'text': "Convergence of Latent Ability Estimators (θ) Over Time",
        'y': 0.93, 'x': 0.5, 'xanchor': 'center', 'yanchor': 'top'
    },
    xaxis_title="Sequence / Item Position (k)",
    yaxis_title="Estimated Ability (θ̂)",
    xaxis=dict(tickmode='linear', tick0=0, dtick=2),
    yaxis=dict(range=[-1, 2]),
    template="plotly_white",
    hovermode="x unified",
    legend=dict(yanchor="top", y=0.15, xanchor="left", x=0.02)
)

fig2.show()

# Q. Bayesian Tracking of Click-Through Rates (CTR) via Conjugate Beta-Binomial Updates

An e-commerce platform wants to optimize its recommendation engine by dynamically estimating the click-through rate (CTR) of a newly launched advertisement. Since user traffic arrives continuously, the platform updates its belief about the advertisement's performance **one impression at a time** rather than waiting for large batch updates.

Let $\Theta = \theta$ represent the true, hidden conversion rate (probability of a click) of the advertisement, where $\theta \in [0, 1]$.

Let $Y_k$ denote a single user's interaction with the advertisement at time step $k$:

$$Y_k =
\begin{cases}
1, & \text{if the user clicks the advertisement}, \\
0, & \text{if the user does not click the advertisement}.
\end{cases}$$

The platform assumes that conditional on the true conversion rate $\Theta = \theta$, each user interaction is an independent Bernoulli trial:

$$P(Y_k = 1 \mid \Theta = \theta) = \theta$$

Let $\mathbf{y}^{(k)} = (y_1, y_2, \dots, y_k)$ represent the **running vector of observed user interactions** up to the current impression step $k$ (where $1 \le k \le n$).

Before observing any data, the platform assigns a flexible **Beta distribution** as the initial prior over the unknown parameter $\Theta$:

$$f_{\Theta}^{(0)}(\theta) = \frac{1}{\mathrm{B}(\alpha_0, \beta_0)} \theta^{\alpha_0 - 1} (1 - \theta)^{\beta_0 - 1} \quad \text{implying} \quad \Theta \sim \text{Beta}(\alpha_0, \beta_0)$$

where $\mathrm{B}(\cdot, \cdot)$ is the Beta function acting as the normalizing constant. Under a sequential framework, the posterior distribution at step $k-1$ serves directly as the prior distribution for step $k$.

---

**Tasks**

**1. Structural Probability and Properties**
Plot the probability density function (PDF) of a $\text{Beta}(\alpha, \beta)$ distribution using Plotly for three distinct parameter pairs:

* Uninformative state: $(\alpha=1, \beta=1)$
* Right-skewed state: $(\alpha=2, \beta=8)$
* Left-skewed state: $(\alpha=8, \beta=2)$

Interpret how changing the balance between $\alpha$ and $\beta$ shifts the center of mass of the density function over the domain $[0, 1]$.

**2. Sequential Likelihood and Joint History**

Write down the mathematical likelihood contribution $L(y_k \mid \theta)$ of a *single* isolated response $y_k$ at step $k$, given the click probability $\theta$. Following this, express the joint likelihood function for the running history vector $\mathbf{y}^{(k)}$.

**3. Closed-Form Analytical Updates (Conjugacy)**

Using Bayes' Theorem, derive the recursive algebraic relationship for the posterior density at step $k$, denoted as $f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)})$. Prove analytically that the posterior remains in the Beta family (**Beta-Binomial Conjugacy**) by explicitly writing down the closed-form update parameters $\alpha_k$ and $\beta_k$ as simple arithmetic updates of $\alpha_{k-1}$, $\beta_{k-1}$, and $y_k$. Also compute the **Posterior Mean** of the latent parameter $\Theta$ at time step $k$ (i.e. $\mathbb{E}[\Theta \mid \mathbf{Y}^{(k)}=\mathbf{y}^{(k)}]$).


**4. Dynamic Shifting Mechanics**

Explain how an observed click ($y_k = 1$) vs. a non-click ($y_k = 0$) shifts the peak of the running density distribution mathematically. Contrast this analytical framework against non-conjugate setups (such as the 2PL IRT model) where numerical grid integration is strictly required.

**5. Running Point Estimators**

State the exact closed-form equations used to evaluate the following point estimates at step $k$ directly from the updated shape parameters $\alpha_k$ and $\beta_k$:

* **Running Posterior Mean** ($\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$)
* **Running Maximum A Posteriori** ($\widehat{\theta}_{\mathrm{MAP}}^{(k)}$)

**6. Performance Tracking and Convergence Analysis**

Suppose the advertisement's true, hidden click-through rate is $\theta_{\text{true}} = 0.35$. Write a Python script to track the performance of your closed-form sequential estimators over a timeline of $n = 100$ impressions:

* **Initialize State:** Set the base prior parameters to $\alpha_0 = 1, \beta_0 = 1$ (representing uniform initial uncertainty).
* **Simulate Responses:** Dynamically generate user responses $y_k \in \{0, 1\}$ at each step by comparing a random draw from a Uniform distribution $U(0,1)$ against $\theta_{\text{true}}$.
* **Track Estimators:** Loop through each step, update $\alpha_k$ and $\beta_k$ analytically, and store the computed values for $\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$ and $\widehat{\theta}_{\mathrm{MAP}}^{(k)}$.
* **Visualize:** Use Plotly to create a single line chart showing the progression of both estimators from step $0$ to $100$. Add a static horizontal reference line at $y = 0.35$ representing $\theta_{\text{true}}$.
* **Analysis:** Explain how the distance between your estimators and $\theta_{\text{true}}$ responds as the sampling size $k$ approaches $100$. What does this imply about the accumulation of evidence over time relative to the choice of the initial prior?

### Visualization of the Beta Distribution

In [ ]:
import numpy as np
import scipy.stats as stats
import plotly.graph_objects as go

# 1. Continuous domain grid evaluation across [0, 1]
grid_domain = np.linspace(0, 1, 500)

# 2. Beta parameter distribution profiles
beta_profiles = [
    {"a_param": 1, "b_param": 1, "label": "Uninformative State: Beta(1,1)", "line_color": "gray", "line_style": "dash"},
    {"a_param": 2, "b_param": 8, "label": "Right-Skewed State: Beta(2,8)", "line_color": "blue", "line_style": "solid"},
    {"a_param": 8, "b_param": 2, "label": "Left-Skewed State: Beta(8,2)", "line_color": "green", "line_style": "solid"}
]

# 3. Instantiate Plotly graphic
fig = go.Figure()

for profile in beta_profiles:
    alpha_val = profile["a_param"]
    beta_val = profile["b_param"]

    # Compute continuous Beta probability density values
    density_curve = stats.beta.pdf(grid_domain, alpha_val, beta_val)

    # Plot profile trace
    fig.add_trace(go.Scatter(
        x=grid_domain,
        y=density_curve,
        mode='lines',
        name=profile["label"],
        line=dict(color=profile["line_color"], dash=profile["line_style"], width=2.5)
    ))

# 4. Canvas styling and formatting
fig.update_layout(
    title={
        'text': "Structural Variations of the Beta(α, β) Probability Density Function",
        'y': 0.93,
        'x': 0.5,
        'xanchor': 'center',
        'yanchor': 'top'
    },
    xaxis_title="Parameter Value (θ)",
    yaxis_title="Probability Density f(θ)",
    xaxis=dict(range=[0, 1], gridcolor='rgba(0,0,0,0.1)'),
    yaxis=dict(gridcolor='rgba(0,0,0,0.1)'),
    template="plotly_white",
    hovermode="x unified",
    legend=dict(
        yanchor="top",
        y=0.95,
        xanchor="center",
        x=0.5,
        bgcolor="rgba(255,255,255,0.7)"
    )
)

# Render figure
fig.show()

### Sequential Likelihood and Joint History

### 1. Likelihood Contribution of a Single Observation

At any given impression $k$, the user response $y_k \in \{0, 1\}$ is modeled as a Bernoulli trial conditioned on the conversion parameter $\theta$. The likelihood function for this individual observation is:

$$L(y_k \mid \theta) = \theta^{y_k} (1 - \theta)^{1 - y_k}$$

* **User Click ($y_k = 1$):** The term reduces to $\theta^1 (1-\theta)^0 = \theta$.
* **User Non-Click ($y_k = 0$):** The term reduces to $\theta^0 (1-\theta)^1 = 1 - \theta$.

---

### 2. Cumulative Joint Likelihood over Observation History

Under the assumption of local independence across user interactions given $\Theta = \theta$, the joint likelihood function for the accumulated response vector $\mathbf{y}^{(k)} = (y_1, y_2, \dots, y_k)$ is formed by taking the product of individual likelihood contributions up to step $k$:

$$L(\mathbf{y}^{(k)} \mid \theta) = \prod_{i=1}^k \theta^{y_i} (1 - \theta)^{1 - y_i} = \theta^{\sum_{i=1}^k y_i} (1 - \theta)^{\sum_{i=1}^k (1 - y_i)}$$

Defining $C_k = \sum_{i=1}^k y_i$ as the cumulative number of clicks, and $k - C_k = \sum_{i=1}^k (1 - y_i)$ as the cumulative number of non-clicks across $k$ total impressions, the overall joint likelihood simplifies to:

$$L(\mathbf{y}^{(k)} \mid \theta) = \theta^{C_k} (1 - \theta)^{k - C_k}$$

---

## Closed-Form Analytical Updates (Beta-Binomial Conjugacy)

### 1. Sequential Bayesian Updating Rule

In a dynamic framework, the posterior distribution computed at step $k-1$ serves as the prior distribution for step $k$. Applying Bayes' Theorem sequentially, the exact posterior density following response $y_k$ is:

$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) = \frac{L(y_k \mid \theta) \cdot f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})}{\int_{0}^{1} L(y_k \mid s) \cdot f_{\Theta \mid \mathbf{Y}^{(k-1)}}(s \mid \mathbf{y}^{(k-1)})\,ds}$$

Omitting the normalizing integral denominator yields the proportional recursive form:

$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto L(y_k \mid \theta) \cdot f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})$$

---

### 2. Proof of Beta Family Conjugacy

Assume the prior state at step $k$ (the posterior from step $k-1$) is distributed according to a Beta density with parameters $\alpha_{k-1}$ and $\beta_{k-1}$:

$$f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)}) = \frac{1}{\mathrm{B}(\alpha_{k-1}, \beta_{k-1})} \theta^{\alpha_{k-1} - 1} (1 - \theta)^{\beta_{k-1} - 1} \propto \theta^{\alpha_{k-1} - 1} (1 - \theta)^{\beta_{k-1} - 1}$$

Substituting the Bernoulli single-observation likelihood into the proportional updating formula:

$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto \left[ \theta^{y_k} (1 - \theta)^{1 - y_k} \right] \cdot \left[ \theta^{\alpha_{k-1} - 1} (1 - \theta)^{\beta_{k-1} - 1} \right]$$

Group terms by adding exponents across common algebraic bases:

$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto \theta^{(\alpha_{k-1} + y_k) - 1} \cdot (1 - \theta)^{(\beta_{k-1} + 1 - y_k) - 1}$$

Because the resulting expression retains the functional form $\theta^{A - 1}(1-\theta)^{B - 1}$, the posterior distribution is analytically proven to belong to the Beta family (**Beta-Binomial Conjugacy**).

---

### 3. Updated Shape Parameters

Equating terms directly yields the deterministic linear updating equations for the shape parameters $(\alpha_k, \beta_k)$:

$$\alpha_k = \alpha_{k-1} + y_k$$

$$\beta_k = \beta_{k-1} + (1 - y_k)$$

Restoring the Beta normalizing constant $\mathrm{B}(\alpha_k, \beta_k)$, the exact normalized posterior probability density function at step $k$ is:

$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) = \frac{1}{\mathrm{B}(\alpha_k, \beta_k)} \theta^{\alpha_k - 1} (1 - \theta)^{\beta_k - 1}$$

### Posterior Mean

In a Beta-Binomial conjugate update framework, the posterior expected value admits a direct analytical solution. Given that the posterior distribution at step $k$ follows $\text{Beta}(\alpha_k, \beta_k)$, its mean is evaluated using the standard expectation of a Beta random variable:

$$\mathbb{E}\left[\Theta \mid \mathbf{Y}^{(k)} = \mathbf{y}^{(k)}\right] = \frac{\alpha_k}{\alpha_k + \beta_k}$$

### Expansion Using Parameter Update Rules

Substituting the closed-form updating identities from Task 3:

* $\alpha_k = \alpha_0 + \sum_{i=1}^k y_i = \alpha_0 + C_k$ *(with $C_k$ representing total recorded clicks up to step $k$)*
* $\beta_k = \beta_0 + \sum_{i=1}^k (1 - y_i) = \beta_0 + (k - C_k)$ *(with $k - C_k$ representing total recorded non-clicks)*

Inserting these state equations into the expectation formula yields:

$$\mathbb{E}\left[\Theta \mid \mathbf{Y}^{(k)} = \mathbf{y}^{(k)}\right] = \frac{\alpha_0 + C_k}{(\alpha_0 + \beta_0) + k}$$

---

**Theoretical Interpretation:**

This expression offers clear statistical intuition:

1. **Compounded Weighted Average:** The posterior expectation acts as a balanced blend combining baseline prior hyper-parameters ($\alpha_0, \beta_0$) with empirical sample observations ($C_k$ clicks across $k$ total impressions).
2. **Asymptotic Convergence:** As total impressions expand toward infinity ($k \to \infty$), the influence of the prior constants ($\alpha_0, \beta_0$) diminishes to zero. The expectation converges directly to $\frac{C_k}{k}$, matching the empirical click proportion (the Maximum Likelihood Estimator).

### Numerical Simulation

In [ ]:
import numpy as np
import scipy.stats as stats
import plotly.graph_objects as go

# Seed setting for reproducible stochastic generation
np.random.seed(42)

# =====================================================================
# SIMULATION SETTINGS & PARAMETERS
# =====================================================================
ground_truth_ctr = 0.35  # Actual click-through conversion rate (35%)
total_impressions = 100
impression_steps = list(range(total_impressions + 1))

# Initial Beta Prior Configuration (Beta(1,1) Uniform Prior)
a_shape = 1
b_shape = 1

# Evaluation domain for continuous probability densities
conversion_grid = np.linspace(0, 1, 500)

# Selected milestone steps to record full density curves
checkpoint_steps = [0, 1, 2, 5, 10, 30, 50, 100]

# Point estimator arrays
posterior_mean_history = [a_shape / (a_shape + b_shape)]
map_mode_history = [0.0]  # Base mode assignment for Uniform(0,1)

# Instantiate Figure 1: Posterior Density Evolution
fig1 = go.Figure()

# Render baseline prior density curve
baseline_density = stats.beta.pdf(conversion_grid, a_shape, b_shape)
fig1.add_trace(go.Scatter(
    x=conversion_grid, y=baseline_density, mode='lines',
    name='Initial Prior: Beta(1,1)',
    line=dict(dash='dash', width=2.5, color='gray')
))

# =====================================================================
# CONJUGATE SEQUENTIAL UPDATE LOOP
# =====================================================================
for step_idx in range(1, total_impressions + 1):
    # Stochastic user interaction draw based on true CTR
    observed_action = 1 if np.random.uniform(0, 1) < ground_truth_ctr else 0

    # EXACT BETA-BINOMIAL CONJUGATE UPDATES:
    a_shape += observed_action
    b_shape += (1 - observed_action)

    # Exact closed-form expectation formula
    est_bayes_mean = a_shape / (a_shape + b_shape)

    # Closed-form mode (MAP) calculation with boundary guards
    if a_shape > 1 and b_shape > 1:
        est_map_mode = (a_shape - 1) / (a_shape + b_shape - 2)
    else:
        est_map_mode = 0.0 if a_shape <= b_shape else 1.0

    posterior_mean_history.append(est_bayes_mean)
    map_mode_history.append(est_map_mode)

    # Capture density PDF curve if step matches a milestone checkpoint
    if step_idx in checkpoint_steps:
        density_curve = stats.beta.pdf(conversion_grid, a_shape, b_shape)
        action_label = "Click" if observed_action == 1 else "No Click"

        fig1.add_trace(go.Scatter(
            x=conversion_grid, y=density_curve, mode='lines',
            name=f"Step {step_idx}: After Event ({action_label}, α={a_shape}, β={b_shape})",
            line=dict(width=2)
        ))

# =====================================================================
# FIGURE 1 LAYOUT: DENSITY PROGRESSION
# =====================================================================
fig1.add_vline(
    x=ground_truth_ctr, line_dash="dot", line_color="red", line_width=2,
    annotation_text=f"True CTR ({ground_truth_ctr})", annotation_position="top right"
)

fig1.update_layout(
    title={
        'text': "Analytical Posterior Density Progression (Beta-Binomial Updates)",
        'y': 0.95, 'x': 0.5, 'xanchor': 'center', 'yanchor': 'top'
    },
    xaxis_title="Conversion Rate Parameter (θ)",
    yaxis_title="Probability Density f(θ | y)",
    template="plotly_white",
    hovermode="x unified",
    legend=dict(
        yanchor="top", y=0.95, xanchor="right", x=0.98,
        bgcolor="rgba(255,255,255,0.7)"
    )
)
fig1.show()

# =====================================================================
# FIGURE 2 LAYOUT: ESTIMATOR CONVERGENCE TIMELINE
# =====================================================================
fig2 = go.Figure()

# True CTR reference baseline
fig2.add_hline(
    y=ground_truth_ctr, line_dash="dash", line_color="red", line_width=2,
    annotation_text=f"True CTR (θ = {ground_truth_ctr})", annotation_position="bottom right"
)

# Posterior expectation trajectory
fig2.add_trace(go.Scatter(
    x=impression_steps, y=posterior_mean_history, mode='lines',
    name='Exact Posterior Mean (Beta Formula)',
    line=dict(color='blue', width=2.5)
))

# MAP mode trajectory
fig2.add_trace(go.Scatter(
    x=impression_steps, y=map_mode_history, mode='lines',
    name='Exact MAP Estimate (Beta Formula)',
    line=dict(color='green', width=1.5, dash='dot')
))

fig2.update_layout(
    title={
        'text': "Analytical Beta-Binomial Conjugate Update Timeline",
        'y': 0.93, 'x': 0.5, 'xanchor': 'center', 'yanchor': 'top'
    },
    xaxis_title="Number of User Impressions (k)",
    yaxis_title="Estimated Conversion Rate (θ̂)",
    template="plotly_white",
    hovermode="x unified",
    legend=dict(yanchor="bottom", y=0.05, xanchor="right", x=0.98)
)
fig2.show()